# One point set, four graphs

Almost every demo in this repository starts the same way: scatter points in the plane, then join them. The stained glass joins them into **Voronoi cells** (`seeds_relaxed`). The flow networks of `demo_vein_models` sit on a **Delaunay** mesh (`leaf_mesh`). Closed venation joins a vein to a source only when they are **relative neighbours**. And a flow network under a steady load settles into a **spanning tree**.

These are not four unrelated ideas. On one point set they form a chain, each graph inside the next:

$$\text{MST} \;\subseteq\; \text{RNG} \;\subseteq\; \text{Gabriel} \;\subseteq\; \text{Delaunay}\;\; (\text{dual of Voronoi}).$$

| graph | keep the link $a$-$b$ when … | where it appears in this repo |
|:--|:--|:--|
| **Delaunay** | some empty circle passes through $a$ and $b$ | lead lines of the stained glass are its dual (Voronoi edges); the mesh of Recipe B |
| **Gabriel** | the circle with diameter $ab$ is empty | (none yet) |
| **RNG** (relative neighbourhood) | no point $c$ is closer to both $a$ and $b$ than they are to each other (the *lune* is empty) | closed venation, Runions et al. eq. 2 |
| **MST** (minimum spanning tree) | the cheapest way to connect everything | the shortest possible vein tree; compare with Recipe B below |

Each test is stricter than the one above it, and each region tested contains the last: the lune contains the diametral disc. That is why the chain nests.

**How to use:** run top to bottom (needs `ipywidgets`).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as w
import scipy.sparse as sp
from IPython.display import display
from matplotlib.collections import LineCollection
from matplotlib.patches import Circle, Polygon
from scipy.sparse.csgraph import dijkstra
from scipy.spatial import ConvexHull, Voronoi

from math_vis import proximity_graphs, lloyd_relaxation
from pattern_formation.networks import relax_moving_graph
from pattern_formation.venation import BG, leaf_outline, leaf_mesh

FAST = "PYTEST_CURRENT_TEST" in os.environ

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "text.color": "#d9e2dc", "axes.labelcolor": "#d9e2dc", "axes.edgecolor": "#3a4a40",
    "xtick.color": "#9fb0a5", "ytick.color": "#9fb0a5",
})
ORDER = ["delaunay", "gabriel", "rng", "mst"]
NAMES = {"delaunay": "Delaunay", "gabriel": "Gabriel", "rng": "RNG", "mst": "MST"}
COLORS = {"delaunay": "#5b6b8c", "gabriel": "#2aa5c9", "rng": "#f59e3b", "mst": "#f4e285"}


def points(n, lloyd, seed):
    p = np.random.default_rng(seed).random((n, 2)) * 1000
    return lloyd_relaxation(p, 1000, 1000, iters=lloyd) if lloyd else p


def lines(ax, p, e, **kw):
    ax.add_collection(LineCollection(p[e], capstyle="round", **kw))


def frame(ax, p):
    pad = 0.03 * np.ptp(p, axis=0).max()
    ax.set_xlim(p[:, 0].min() - pad, p[:, 0].max() + pad)
    ax.set_ylim(p[:, 1].min() - pad, p[:, 1].max() + pad)
    ax.set_aspect("equal")
    ax.axis("off")

## 1. The four graphs, one at a time

Each panel draws its graph bold and the previous (larger) graph faint, so you see which links each test throws away. The Delaunay panel shows the Voronoi cells too: one Delaunay link for every pair of cells that share a wall, the same walls that become lead in the stained glass. In the Gabriel panel one rejected link is shown with its disc and the point inside it; in the RNG panel, one rejected link with its lune.

In [ ]:
def first_rejected(p, kept_in, kept):
    s = set(map(tuple, kept))
    gone = [e for e in kept_in if tuple(e) not in s]
    centre = p.mean(axis=0)
    return min(gone, key=lambda e: np.linalg.norm(p[e].mean(axis=0) - centre))


def lune(a, b, n=60):
    L = np.linalg.norm(b - a)
    th = np.arctan2(*(b - a)[::-1])
    arc1 = a + L * np.column_stack([np.cos(th + np.linspace(-np.pi / 3, np.pi / 3, n)),
                                    np.sin(th + np.linspace(-np.pi / 3, np.pi / 3, n))])
    arc2 = b + L * np.column_stack([np.cos(th + np.pi + np.linspace(-np.pi / 3, np.pi / 3, n)),
                                    np.sin(th + np.pi + np.linspace(-np.pi / 3, np.pi / 3, n))])
    return np.vstack([arc1, arc2])


def show_chain(p):
    g = proximity_graphs(p)
    fig, axs = plt.subplots(1, 4, figsize=(16, 4.4))
    vor = Voronoi(p)
    for k, (ax, name) in enumerate(zip(axs, ORDER)):
        if k == 0:
            walls = [r for r in vor.ridge_vertices if -1 not in r]
            lines(ax, vor.vertices, np.array(walls), colors="#8e2a6a", linewidths=0.9)
        else:
            lines(ax, p, g[ORDER[k - 1]], colors=COLORS[ORDER[k - 1]], linewidths=0.8, alpha=0.3)
        lines(ax, p, g[name], colors=COLORS[name], linewidths=1.6)
        ax.scatter(*p.T, s=6, color="#d9e2dc", zorder=3)
        if name in ("gabriel", "rng"):
            i, j = first_rejected(p, g[ORDER[k - 1]], g[name])
            a, b = p[i], p[j]
            if name == "gabriel":
                ax.add_patch(Circle((a + b) / 2, np.linalg.norm(b - a) / 2, fill=False, ec="#e0453a", lw=1.4))
            else:
                ax.add_patch(Polygon(lune(a, b), fill=False, ec="#e0453a", lw=1.4))
            ax.plot(*p[[i, j]].T, color="#e0453a", lw=2, ls="--")
        ax.set_title(f"{NAMES[name]}: {len(g[name])} links", color="#d9e2dc")
        frame(ax, p)
    plt.tight_layout()
    plt.show()


show_chain(points(70 if FAST else 90, 0, 3))

## 2. All four in one picture

Draw every link once, as nested bands: a link in the MST gets all four colours, a link only in Delaunay gets just the thin dark line. Because every band sits inside the next wider one, the picture *is* the chain: you will never see a yellow band without orange around it.

The **Lloyd** slider moves the points toward the centres of their Voronoi cells, the same step that makes the stained-glass cells even. Watch what that does to the chain.

In [ ]:
def degrees(p, g):
    inner = (np.abs(p - p.mean(axis=0)) < 0.4 * np.ptp(p, axis=0)).all(axis=1)  # away from the edge
    return {k: np.bincount(g[k].ravel(), minlength=len(p))[inner].mean() for k in ORDER}


def show_nested(n, lloyd, seed):
    p = points(n, lloyd, seed)
    g = proximity_graphs(p)
    fig, ax = plt.subplots(figsize=(7.5, 7.5))
    for name, lw in zip(ORDER[::-1], (12.0, 8.0, 4.5, 1.5)):
        lines(ax, p, g[name], colors=COLORS[name], linewidths=lw)
    ax.scatter(*p.T, s=8, color="#0d1411", zorder=3)
    frame(ax, p)
    ax.legend(handles=[plt.Line2D([], [], color=COLORS[k], lw=4, label=NAMES[k]) for k in ORDER],
              loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=4, frameon=False)
    plt.show()
    d = degrees(p, g)
    print("mean links per node (away from the edge): "
          + "   ".join(f"{NAMES[k]} {d[k]:.2f}" for k in ORDER))


ctrl_n = dict(
    n=w.IntSlider(value=150, min=30, max=600, step=10, description="points"),
    lloyd=w.IntSlider(value=0, min=0, max=30, description="Lloyd steps"),
    seed=w.IntSlider(value=3, min=0, max=20, description="seed"),
)
display(w.interactive(show_nested, **ctrl_n))
if FAST:
    show_nested(60, 2, 3)

**Question 1.** With random points (Lloyd 0), how many links per node does each graph have?

<details><summary>Answer</summary>

Delaunay 6, Gabriel 4, RNG about 2.55, MST 2 (the next cell measures them on 4000 points). Delaunay is 6 because a triangulation of $N$ points with $h$ on the hull has exactly $3N-3-h$ links (Euler's formula), so twice that over $N$ tends to 6. The MST is a tree, $N-1$ links, so 2. Gabriel 4 is a known result for random points (Matula and Sokal [2]); RNG has no closed form that I know of.

</details>

**Question 2.** Raise the Lloyd steps. Which graphs change and which do not?

<details><summary>Answer</summary>

Delaunay and MST stay at 6 and 2: they are fixed by counting. Gabriel climbs from 4 to about 5.6 and RNG from 2.55 to about 3.25 after 30 steps. Even spacing makes Delaunay triangles closer to equilateral, and an equilateral triangle passes the Gabriel test (its third corner is $\tfrac{\sqrt3}{2}L$ from the midpoint, outside the disc of radius $L/2$). On a perfect hexagonal lattice Gabriel would equal Delaunay. **Open:** Lloyd stops near 5.6, not 6. Is that lattice defects the relaxation never removes, or the coarse grid `lloyd_relaxation` uses?

</details>

## 3. Numbers the calculations predicted

1. Delaunay has exactly $3N - 3 - h$ links ($h$ points on the convex hull).
2. MST has exactly $N - 1$ links.
3. Every graph lies inside the next (checked link by link).
4. Mean links per node for random points: Delaunay → 6, Gabriel → 4.

These are tests in `tests/test_math_tools.py`, where RNG and Gabriel are also checked against a brute-force search over every pair.

In [ ]:
def check_numbers(n):
    print(f"{'points':>10s}{'h':>6s}{'Delaunay':>10s}{'3N-3-h':>8s}{'MST':>6s}{'N-1':>6s}  nested   "
          + "  ".join(f"{NAMES[k]:>8s}" for k in ORDER))
    for name, lloyd in [("random", 0), ("Lloyd 3", 3), ("Lloyd 30", 30)]:
        p = points(n, lloyd, 1)
        g = proximity_graphs(p)
        h = len(ConvexHull(p).vertices)
        s = [set(map(tuple, g[k])) for k in ORDER]
        nested = s[3] <= s[2] <= s[1] <= s[0]
        d = degrees(p, g)
        print(f"{name:>10s}{h:6d}{len(g['delaunay']):10d}{3 * n - 3 - h:8d}{len(g['mst']):6d}{n - 1:6d}  {str(nested):6s}   "
              + "  ".join(f"{d[k]:8.2f}" for k in ORDER))


check_numbers(500 if FAST else 4000)

## 4. Where do the veins sit?

Recipe B of `demo_vein_models` (adaptive flow, steady load, $\gamma < 1$) settles into a spanning tree of the leaf mesh. A spanning tree is not automatically *the minimum* spanning tree. So which tree is it?

**Calculation.** A settled pipe has $C = (Q^2/\nu)^{1/(\gamma+1)}$. At $\gamma = 1$ that is $C = |Q|/\sqrt\nu$, and the energy $E = \tfrac12\sum \ell\,(Q^2/C + \nu C)$ becomes $\sqrt\nu\sum \ell\,|Q|$: total flow times distance. With one source and every node draining, that is smallest when each node is fed along its **shortest path** from the base. So near $\gamma = 1$ the veins should form the *shortest-path tree*, the opposite end from the MST, which makes the total length as short as possible and ignores how far water travels.

The cell draws the three trees on one mesh and measures the mean path from the base along each tree, divided by the shortest distance through the mesh (1 means every node is fed by a shortest path).

In [ ]:
def tree_paths(xy, e, base):
    L = np.linalg.norm(xy[e[:, 0]] - xy[e[:, 1]], axis=1)
    A = sp.coo_matrix((L, (e[:, 0], e[:, 1])), shape=(len(xy), len(xy)))
    return dijkstra(A, directed=False, indices=base, return_predecessors=True)


def compare_trees(rows, seed):
    poly = leaf_outline("ovate")
    xy, edges, base = leaf_mesh(poly, rows, seed=seed)
    N = len(xy)
    length = np.linalg.norm(xy[edges[:, 0]] - xy[edges[:, 1]], axis=1)
    sinks = np.array([i for i in range(N) if i != base])
    dist, pred = tree_paths(xy, edges, base)
    spt = np.sort(np.array([(i, pred[i]) for i in range(N) if i != base]), axis=1)
    trees = {"MST": proximity_graphs(xy)["mst"], "shortest-path tree": spt}
    for gamma in (0.25, 0.5, 0.9):
        C = relax_moving_graph(edges, N, base, sinks, 0.0, gamma=gamma, dt=0.5,
                               steps=200 if FAST else 800, length=length)
        trees[f"flow, gamma {gamma}"] = edges[C > 1e-4 * C.max()]
    S = {k: set(map(tuple, np.sort(v, axis=1))) for k, v in trees.items()}
    print(f"{'tree':22s}{'links':>7s}{'length':>8s}{'path / shortest':>17s}{'shared with MST':>17s}{'with SPT':>10s}")
    for k, e in trees.items():
        tot = np.linalg.norm(xy[e[:, 0]] - xy[e[:, 1]], axis=1).sum()
        ratio = tree_paths(xy, e, base)[0].mean() / dist.mean()
        print(f"{k:22s}{len(e):7d}{tot:8.2f}{ratio:17.3f}{len(S[k] & S['MST']) / len(e):17.2f}"
              f"{len(S[k] & S['shortest-path tree']) / len(e):10.2f}")
    fig, axs = plt.subplots(1, 3, figsize=(12, 5))
    for ax, (k, col) in zip(axs, [("MST", COLORS["mst"]), ("shortest-path tree", COLORS["gabriel"]),
                                  ("flow, gamma 0.5", "#6fae3f")]):
        ax.fill(*poly.T, color="#16231b")
        lines(ax, xy, trees[k], colors=col, linewidths=1.6)
        ax.scatter(*xy[base], s=40, color="#e0453a", zorder=3)
        ax.set_title(k, color="#d9e2dc")
        frame(ax, xy)
    plt.tight_layout()
    plt.show()


compare_trees(12 if FAST else 24, 0)

**What held.** The flow trees feed every node along an almost shortest path: path ratio 1.006, 1.002 and 1.000 at $\gamma$ = 0.25, 0.5 and 0.9 (24 rows, seeds 0 to 2), and they share 84 to 100 % of their links with the shortest-path tree but only about 44 % with the MST. The MST is about 22 to 25 % shorter in total but its paths are 2.5 times longer. So a leaf that drains everywhere builds the *travel-time* tree, not the *material* tree; lowering $\gamma$ moves it a little toward sharing trunks. This is a test in `pattern_formation/tests/test_venation.py`.

**Not exact.** At $\gamma = 1$ exactly, comparing links one by one is noisy: the jittered mesh has many almost equal paths, and at $\gamma = 1$ the unused links decay slowly. The path ratio is the clean number.

**Open.** Where does closed venation (Recipe A) sit? Its nodes are not a fixed point set, so the chain has to be built on its node positions after growth. Does it keep only RNG links, as its joining rule suggests?

## 5. Back to the glass

The stained glass and the veins are two views of the same object. The glass draws **faces** (cells), the veins draw **links**. For a planar network, Euler's formula ties the two together: faces = loops + 1 (the +1 is the outside). `demo_vein_glass` uses this. It draws a vein network as leaded glass, the veins as lead, the areoles as panes, and the number of panes it finds is exactly the number of loops plus one.

## Sources

[1] A. Runions et al., *Modeling and visualization of leaf venation patterns*, SIGGRAPH 2005. The relative-neighbour rule of closed venation (eq. 2). See `demo_vein_models` for the full list.

[2] D. W. Matula, R. R. Sokal, *Properties of Gabriel graphs relevant to geographic variation research and the clustering of points in the plane*, Geographical Analysis 12, 205-222 (1980). Mean Gabriel degree 4 for random points.

[3] G. T. Toussaint, *The relative neighbourhood graph of a finite planar set*, Pattern Recognition 12, 261-268 (1980). Introduces the RNG and proves MST ⊆ RNG ⊆ Delaunay.

What I read: none of [2] and [3]; they are the standard references for these results. Every number in this notebook is measured here, and the chain is checked link by link against a brute-force search.